# Diffusion MRI Preprocessing Workflow Example

For this example you will need:
- Diffusion weighted images (DWIs) in the NIFTI format (including bvals and bvecs):
    - Image with DWIs and b0s
    - A reverse phase encoding b0 with extra DWIs
    - Access to phase encoding direction, partial Fourier fraction, and readout time.
- These packages available in this session:
    - `dwi2niix` (https://github.com/rordenlab/dcm2niix): for DICOM conversion
    - DesignerV2 (https://github.com/NYU-DiffusionMRI/DESIGNER-v2): denoising and Gibbs correction
    - FSL (https://fsl.fmrib.ox.ac.uk) 
        - Susceptibility correction with `TOPUP`
        - Masking with `bet`
        - Eddy distortion correction with `EDDY`
        - Visualization with `fsleyes`
        - Other utilities
    - MRTrix3 (https://mrtrix.org/)
        - Bias field correction with `dwibiascorrect`
        - DWI Masking with `dwi2mask`
        - Other utilities
    - ANTs (https://github.com/antsx/ants)
        - `N4BiasFieldCorrection` needed for `dwibiascorrect ants`

This example uses ADNI data (ID: 052_S_10387_20250102_A4_DWI_S127) from the Image and Data Archive (IDA). 

In [ ]:
%cd "/your/directory/here"

## 00. Convert DICOM to NIFTI

Use `dcm2niix` (https://github.com/rordenlab/dcm2niix) to convert DICOMs to NIFTIs using the bids format (`-b y`). Then extract the tracking parameters for the phase encoding (PE) direction and readout time in an aqcuisitions file (`acqparams.txt`) with `mrconvert`.

In [ ]:
%%bash

# Files might come in the DICOM format. If so, use dcm2niix to convert them to NIFTI format
# Convert the PA DWI to NIFTI
dcm2niix \
    -z "y" -b "y" \
    -f "dwi_PA" \
    -o "00_raw_data"\
    "00_raw_data/DCM_PA"

# Convert the AP DWI to NIFTI
dcm2niix \
    -z "y" -b "y" \
    -f "dwi_AP" \
    -o "00_raw_data" \
    "00_raw_data/DCM_AP"

# Extract metadata needed for TOPUP and EDDY 
mrconvert "00_raw_data/dwi_AP.nii.gz" "00_raw_data/dwi_AP.mif" \
    -json_import "00_raw_data/dwi_AP.json" -fslgrad "00_raw_data/dwi_AP.bvec" "00_raw_data/dwi_AP.bval" \
    -export_pe_eddy "00_raw_data/acqparams_AP.txt" "00_raw_data/index_AP.txt"
    
# Extract metadata needed for TOPUP and EDDY 
mrconvert "00_raw_data/dwi_PA.nii.gz" "00_raw_data/dwi_PA.mif" \
    -json_import "00_raw_data/dwi_PA.json" -fslgrad "00_raw_data/dwi_PA.bvec" "00_raw_data/dwi_PA.bval" \
    -export_pe_eddy "00_raw_data/acqparams_PA.txt" "00_raw_data/index_PA.txt"

Look in the JSON:
- `SeriesDescription`: find the slice axis
- `PhaseEncodingDirection`: deGibbs, TOPUP, and EDDY
- `TotalReadoutTime`: TOPUP & EDDY
- `PartialFourier`: deGibbs

In [47]:
%%bash
printf "File header:\n"
fslhd "00_raw_data/dwi_PA.nii.gz"
printf "\nAcquisition parameters:\n"
cat "00_raw_data/dwi_PA.json"

File header:
filename	00_raw_data/dwi_PA.nii.gz

sizeof_hdr	348
data_type	INT16
dim0		4
dim1		116
dim2		116
dim3		81
dim4		127
dim5		1
dim6		1
dim7		1
vox_units	mm
time_units	s
datatype	4
nbyper		2
bitpix		16
pixdim0		-1.000000
pixdim1		2.000000
pixdim2		2.000000
pixdim3		2.000000
pixdim4		3.400000
pixdim5		0.000000
pixdim6		0.000000
pixdim7		0.000000
vox_offset	352
cal_max		0.000000
cal_min		0.000000
scl_slope	1.000000
scl_inter	0.000000
phase_dim	2
freq_dim	1
slice_dim	3
slice_name	Unknown
slice_code	0
slice_start	0
slice_end	0
slice_duration	0.000000
toffset		0.000000
intent		Unknown
intent_code	0
intent_name	
intent_p1	0.000000
intent_p2	0.000000
intent_p3	0.000000
qform_name	Scanner Anat
qform_code	1
qto_xyz:1	-2.000000 0.000000 -0.000000 114.644043 
qto_xyz:2	0.000000 2.000000 -0.000000 -101.796631 
qto_xyz:3	0.000000 0.000000 2.000000 -88.135597 
qto_xyz:4	0.000000 0.000000 0.000000 1.000000 
qform_xorient	Right-to-Left
qform_yorient	Posterior-to-Anterior
qform_zorient	Inferior-

In [ ]:
%%bash
# View the raw NIFTIs using FSLs `fslview`
fsleyes \
    "00_raw_data/dwi_PA.nii.gz" \
    "00_raw_data/dwi_AP.nii.gz" &

![00_raw_data](figures/00_raw_data-PA.png)

*Raw DWI showing noise, inter-slice motion and blinding.*

## 01. Denoising

Reduce random noise in the image using `designer`:

- `-denoise`: MP-PCA denoising.
- `-extent 5,5,5`: patch size in voxels.
- `-algorithm cordero-grande`: noise estimator.
- `-fslbvec`, `-fslbval`: gradient directions and b-values.
- `-bids`: JSON metadata file.
- `-pf`, `-pe_dir`: partial Fourier fraction (e.g. `1` or `7/8`) and main NIfTI PE direction (e.g. `j` or `j-`). DESIGNER requires these even for denoising.
- `-nocleanup`, `-scratch`: keep intermediate files under `01_denoise/`.

`mrconvert` exports the image to NIfTI. `mrcalc` is used to subtract the denoised (`dwi_dn.nii.gz`) and original images. The subtracted residuals (`dwi_dn_residual.nii.gz`) should look like noise, without recognizable anatomy.


In [ ]:
%%bash

mkdir -p "01_denoise"

## Denoise for AP run
docker run --rm --platform linux/amd64 \
    -v "$(pwd)/00_raw_data:/data/00_raw_data:ro" \
    -v "$(pwd)/01_denoise:/data/01_denoise" \
    -w "/data" \
    nyudiffusionmri/designer2:v2.0.16 \
    designer "/data/00_raw_data/dwi_AP.nii.gz" \
    "/data/01_denoise/dwi_AP_dn.mif" \
    -denoise -extent 5,5,5 -algorithm cordero-grande \
    -fslbvec "/data/00_raw_data/dwi_AP.bvec" \
    -fslbval "/data/00_raw_data/dwi_AP.bval" \
    -bids "/data/00_raw_data/dwi_AP.json" \
    -nocleanup -scratch "/data/01_denoise"

# Denoise for PA run
docker run --rm --platform linux/amd64 \
    -v "$(pwd)/00_raw_data:/data/00_raw_data:ro" \
    -v "$(pwd)/01_denoise:/data/01_denoise" \
    -w "/data" \
    nyudiffusionmri/designer2:v2.0.16 \
    designer "/data/00_raw_data/dwi_PA.nii.gz" \
    "/data/01_denoise/dwi_PA_dn.mif" \
    -denoise -extent 5,5,5 -algorithm cordero-grande \
    -fslbvec "/data/00_raw_data/dwi_PA.bvec" \
    -fslbval "/data/00_raw_data/dwi_PA.bval" \
    -bids "/data/00_raw_data/dwi_PA.json" \
    -nocleanup -scratch "/data/01_denoise"

# Calculate the noise by subtracting before and after
mrconvert "01_denoise/dwi_PA_dn.mif" "01_denoise/dwi_PA_dn.nii.gz"
mrcalc "00_raw_data/dwi_PA.nii.gz" \
    "01_denoise/dwi_PA_dn.nii.gz" -subtract "01_denoise/dwi_PA_dn_residual.nii.gz"

# Calculate the noise by subtracting before and after
mrconvert "01_denoise/dwi_AP_dn.mif" "01_denoise/dwi_AP_dn.nii.gz"
mrcalc "00_raw_data/dwi_AP.nii.gz" \
    "01_denoise/dwi_AP_dn.nii.gz" -subtract "01_denoise/dwi_AP_dn_residual.nii.gz"

In [ ]:
%%bash

# View the inputs, outputs and residuals
fsleyes \
    "00_raw_data/dwi_PA.nii.gz" \
    "01_denoise/dwi_PA_dn.nii.gz" \
    "01_denoise/dwi_PA_dn_residual.nii.gz" &

1. <img src="figures/01_denoise.gif" width="100%"></img>
2. ![Add figure](figures/01_denoise-residual.png)

*Removing noise: (1.a) noisy b0; (1.b) denoised b0; and (2) subtracted residuals (noise map).*

## 02. Gibbs ringing correction

Reduce ripples near tissue boundaries: 
- `-degibbs` applies the correction
- `-pf` gives the Fourier fraction
- `-pe_dir` gives the main PE direction. 

The residual map is the denoised image minus the Gibbs corrected image, and it should look like an outline of anatomical boundaries with sharp contrast (e.g. around the ventricles). Look for reduced ringing while preserving tissue edges.

In [49]:
%%bash

# Find the PF and PE direction for PA run
printf "PA:"
grep -E '"PartialFourier"|"PhaseEncodingDirection"' "00_raw_data/dwi_PA.json"

# Find the PF and PE direction for AP run
printf "\nAP:"
grep -E '"PartialFourier"|"PhaseEncodingDirection"' "00_raw_data/dwi_AP.json"

PA:	"PartialFourier": 0.75,
	"PhaseEncodingDirection": "j",

AP:	"PartialFourier": 0.75,
	"PhaseEncodingDirection": "j-",


In [ ]:
%%bash

mkdir -p "02_degibbs"

# Remove gibbs ringing for PA run
docker run --rm --platform linux/amd64 \
    -v "$(pwd)/01_denoise:/data/01_denoise:ro" \
    -v "$(pwd)/02_degibbs:/data/02_degibbs" \
    -w "/data" \
    nyudiffusionmri/designer2:v2.0.16 \
    designer "01_denoise/dwi_PA_dn.mif" "02_degibbs/dwi_PA_dn_dg.mif" \
    -degibbs -pf 0.75 -pe_dir "j" \
    -fslbvec "01_denoise/dwi_PA_dn.bvec" -fslbval "01_denoise/dwi_PA_dn.bval" \
    -nocleanup -scratch "02_degibbs"

# Remove gibbs ringing for AP run
docker run --rm --platform linux/amd64 \
    -v "$(pwd)/01_denoise:/data/01_denoise:ro" \
    -v "$(pwd)/02_degibbs:/data/02_degibbs" \
    -w "/data" \
    nyudiffusionmri/designer2:v2.0.16 \
    designer "01_denoise/dwi_AP_dn.mif" "02_degibbs/dwi_AP_dn_dg.mif" \
    -degibbs -pf 0.75 -pe_dir "j-" \
    -fslbvec "01_denoise/dwi_AP_dn.bvec" -fslbval "01_denoise/dwi_AP_dn.bval" \
    -nocleanup -scratch "02_degibbs"

# Calculate the residual ringing for the PA run by subtracting before and after
mrconvert "02_degibbs/dwi_PA_dn_dg.mif" "02_degibbs/dwi_PA_dn_dg.nii.gz"
mrcalc "01_denoise/dwi_PA_dn.nii.gz" "02_degibbs/dwi_PA_dn_dg.nii.gz" \
    -subtract "02_degibbs/dwi_PA_dn_dg_residual.nii.gz"

# Calculate the residual ringing for the AP run by subtracting before and after
mrconvert "02_degibbs/dwi_AP_dn_dg.mif" "02_degibbs/dwi_AP_dn_dg.nii.gz"
mrcalc "01_denoise/dwi_AP_dn.nii.gz" "02_degibbs/dwi_AP_dn_dg.nii.gz" \
    -subtract "02_degibbs/dwi_AP_dn_dg_residual.nii.gz"

In [ ]:
%%bash

# View the inputs, outputs and residuals
fsleyes \
    "01_denoise/dwi_PA_dn.nii.gz" \
    "02_degibbs/dwi_PA_dn_dg.nii.gz" \
    "02_degibbs/dwi_PA_dn_dg_residual.nii.gz"

1. <img src="figures/02_degibbs.gif" width="100%"></img>
2. ![Add figure](figures/02_degibbs-residual.png)

*Gibbs ringing correction: (1.a) b0 before correction; (1.b) b0 after correction; (2) residuals.*


## 03. TOPUP inputs

Extract the b0 images and concatenate primary and reverse phase encoding images.
- `-bzero`: concatenated b0 volumes.
- `-config BZeroThreshold 50`: set maximum b0 threshold.
- `-fslgrad`: import b-vectors followed by b-values.
- `-t`: concatenate along the time dimension.

In [ ]:
%%bash

mkdir -p "03_topup_inputs"

# Extract the b0 volumes from the PA image
dwiextract "02_degibbs/dwi_PA_dn_dg.nii.gz" "03_topup_inputs/b0s_PA.nii.gz" \
    -fslgrad "02_degibbs/dwi_PA_dn_dg.bvec" "02_degibbs/dwi_PA_dn_dg.bval" \
    -bzero -config BZeroThreshold 50

# Extract the b0 volumes from the AP image
dwiextract "02_degibbs/dwi_AP_dn_dg.nii.gz" "03_topup_inputs/b0_AP.nii.gz" \
    -fslgrad "02_degibbs/dwi_AP_dn_dg.bvec" "02_degibbs/dwi_AP_dn_dg.bval" \
    -bzero -config BZeroThreshold 50

# Merge the b0 volumes so the PA volumes are first and AP volumes are second
fslmerge -t "03_topup_inputs/b0_PAAP.nii.gz" \
    "03_topup_inputs/b0_PA.nii.gz" "03_topup_inputs/b0_AP.nii.gz"

TOPUP needs a file with one acquisition parameter row per b0. Each row is `PE_x PE_y PE_z readout_seconds`. 

For this example we have 13 PA b0s and 1 AP b0 volume per phase encoding direction (14 rows). For the j-axis encoding, replace the sign fields with `1` or `-1` (all other axes get 0) and the readout fields with times in seconds. Confirm the actual signs rather than assuming them from the AP/PA names.  

We already exported the acquisition parameters in step `00. Convert DICOM to NIFTI`. We can build our TOPUP input using these.

In [ ]:
%%bash

# Write an acquisition params file with one acquisition parameter row per volume in merged b0 PAAP image
#get the number of volumes in each b0 image
nPA=$(fslval "03_topup_inputs/b0s_PA.nii.gz" dim4)
nAP=$(fslval "03_topup_inputs/b0_AP.nii.gz" dim4)

: > "03_topup_inputs/acqparams.txt"
for ((i=1;i<=nPA;i++)); do
    cat "00_raw_data/acqparams_PA.txt" >> "03_topup_inputs/acqparams.txt"
done

for ((i=1;i<=nAP;i++)); do
cat "00_raw_data/acqparams_AP.txt" >> "03_topup_inputs/acqparams.txt"
done

# View your file which will input into TOPUP
# i j k readout (e.g. 0 1 0 0.0632)
echo "Length of file is: $(wc -l < 03_topup_inputs/acqparams.txt)"
echo "i j k readout"
echo "_____________"
cat "03_topup_inputs/acqparams.txt"

Length of file is:       14
i j k readout
_____________
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 1 0 0.0632
0 -1 0 0.0632


<img src="figures/03_topup_inputs.gif" width="100%"></img>

*TOPUP b0 inputs: (a) b0 with blip-up (PA); (b) b0 with blip-down (AP).*

## 04. Susceptibility distortion correction

Estimate the distortion field and unwarp the b0 images with FSL's `topup`.

- `--imain`, `--datain`: b0 stack and acquisition file.
- `--config=b02b0_1.cnf`: TOPUP settings. If slice number is odd then use b02b0_1.cnf else b02b0.cnf.
- `--out`: output basename, also passed to EDDY later.
- `--iout`: corrected b0 images.
- `--fout`: estimated field in Hz.


In [ ]:
%%bash

mkdir -p "04_topup_outputs"

# Getting the dimension from the header (dim3)
# If this value is odd then use b02b0_1.cnf else b02b0.cnf
printf "Number of slices: "
echo $(fslval "00_raw_data/dwi_PA.nii.gz" dim3)

# get the susceptibility distortion field for eddy
topup --verbose \
    --imain="03_topup_inputs/b0_PAAP.nii.gz" \
    --datain="03_topup_inputs/acqparams.txt" \
    --config="config/b02b0_1.cnf" \
    --out="04_topup_outputs/topup" \
    --iout="04_topup_outputs/b0_PAAP_uw" \
    --fout="04_topup_outputs/topup_field"

Number of slices: 81


In [ ]:
%%bash

# View the distortion field over the corrected and uncorrected images
fsleyes \
    "04_topup_outputs/b0_PAAP_uw.nii.gz" \
    "03_topup_inputs/b0_PAAP.nii.gz" \
    "04_topup_outputs/topup_field.nii.gz"

<img src="figures/04_topup_output.gif" width="100%"></img>

*TOPUP distortion correction with distortion field overlaid: (a) distorted b0 image (PA); (b) corrected b0 image.*

In [ ]:
%%bash
# Create a QC figure of the susceptibility correction
cd "04_topup_outputs"
slicesdir -p "../04_topup_outputs/b0_PAAP_uw_mean.nii.gz" "../03_topup_inputs/b0_PA.nii.gz"

<img src="figures/03_topup_outputs-b0_PA.png" width="100%"></img>

*FSL's `slicedir` output with distorted DWI (PA) and mean corrected b0 outline overlaid.*

## 05. Brain mask

Make mean b0 from topup, then use FLS's `bet -m` to create a brain mask, with `-f 0.3` setting the fractional intensity threshold.

Adjust the threshold after viewing the mask if over- or under-estimated. Higher and lower values are less and more strict, respectively. Slight overestimation is better than underestimation for EDDY input.

In [ ]:
%%bash
mkdir -p "05_brain_mask"

# create mean b0 from the unwarped b0s to use for masking
mrmath "04_topup_outputs/b0_PAAP_uw.nii.gz" mean -axis 3 "04_topup_outputs/b0_PAAP_uw_mean.nii.gz"

# create a mask on the corrected mean b0 from topup
bet "04_topup_outputs/b0_PAAP_uw_mean.nii.gz" "05_brain_mask/b0_brain" -m -f 0.3

# creating a qc figure with multiple slices of the corrected mean b0 and the transparent mask overlaid.
# https://github.com/ENIGMA-git/ENIGMA-DTI-Preprocessing-Guidelines/blob/master/scripts/qc_image_generation.py
"../bin/qc_image_generation.py" \
    -o "05_brain_mask/brain_mask_qc.png" \
    -i "04_topup_outputs/b0_PAAP_uw_mean.nii.gz" \
    -m "05_brain_mask/b0_brain_mask.nii.gz" \
    -x 30 40 50 60 70 80 -y 30 40 50 60 70 80 -z 20 40 60

In [ ]:
%%bash
fsleyes \
    "04_topup_outputs/b0_PAAP_uw_mean.nii.gz" \
    "05_brain_mask/b0_brain_mask.nii.gz"

![Add figure](figures/05_brain_mask-qc.png)

*Transparent brain mask overlaid on the mean corrected b0.*


## 06. EDDY inputs

Concatenate the PA and AP images using `fslmerge` in the time dimension `-t`. Concatenate the bvals and bvecs in the same order. Create a new parameters file with one row per image acquisition parameter profile (in our case 2 rows). Make a file called `index.txt` that matches the acquisition parameter row to the volume using the row index.

In [68]:
%%bash

# mkdir -p "06_eddy_inputs"

# merge images so PA volumes are first and AP volumes are second in time (-t)
fslmerge -t "06_eddy_inputs/dwi_PAAP.nii.gz" \
    "02_degibbs/dwi_PA_dn_dg.nii.gz" "02_degibbs/dwi_AP_dn_dg.nii.gz"

# merge the FSL files (PA+AP) as columns with a space (-d ' ') between them 
# make sure they are in the same order as the images
paste -d ' ' "00_raw_data/dwi_PA.bval" "00_raw_data/dwi_AP.bval" >> "06_eddy_inputs/dwi_PAAP.bval"
paste -d ' ' "00_raw_data/dwi_PA.bvec" "00_raw_data/dwi_AP.bvec" >> "06_eddy_inputs/dwi_PAAP.bvec"

# make a acquisition parameter file with a row per phase encoding direction (PA\AP)
: > "06_eddy_inputs/acqparams.txt"
cat "00_raw_data/acqparams_PA.txt" >> "06_eddy_inputs/acqparams.txt"
cat "00_raw_data/acqparams_AP.txt" >> "06_eddy_inputs/acqparams.txt"

# find the number of volumes in each image using the number of lines in the bval table
# wc -w count the number of words
nPA=$(wc -w < "00_raw_data/dwi_PA.bval")
nAP=$(wc -w < "00_raw_data/dwi_AP.bval")

# Create the index file that tells eddy which parameter to use for each volume in the concatenated image
: > "06_eddy_inputs/index.txt"

# Starts with 1xnPA
for ((i=1; i<=$nPA;i++)); do
    printf "1 " >> "06_eddy_inputs/index.txt"
done

# Ends with 2xnAP
for ((i=1; i<=$nAP;i++)); do
    printf "2 " >> "06_eddy_inputs/index.txt"
done

# it should be the same length as the concatenated image
echo "Index file length: $(wc -w < 06_eddy_inputs/index.txt)"
echo "Image length:           $(fslval 06_eddy_inputs/dwi_PAAP.nii.gz dim4)"

printf "\nAcquisition Parameters (two rows)\n"
cat "06_eddy_inputs/acqparams.txt"

printf "\nIndex File (one row)\n"
cat "06_eddy_inputs/index.txt" | xargs -n 25


Index file length:      134
Image length:           134 

Acquisition Parameters (two rows)
0 1 0 0.0632
0 -1 0 0.0632

Index File (one row)
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 2 2 2 2 2 2 2


## 07a. Motion and eddy-current correction

Correct motion and eddy-current distortion while applying the TOPUP field.

- `--imain`, `--mask`: input DWI and brain mask.
- `--acqp`, `--index`: acquisition table and per-volume row indices.
- `--bvecs`, `--bvals`: gradient files.
- `--topup`, `--out`: TOPUP input and EDDY output basenames.
- `--residuals`, `--cnr_maps`: save images for quality assessment.
- `--repol`: interpolate outlier slices.
- `--resamp=jac`: Jacobian-modulated resampling.
- `--flm=quadratic`: quadratic eddy-current field model.
- `--niter=8`, `--fwhm=10,6,4,2,0,0,0,0`: iteration count and estimation smoothing schedule in mm.

Use `dwi_eddy.eddy_rotated_bvecs` for subsequent processing and modeling.

This run will fix between volume motion, susceptibility distortion and drop out, but will not fix within volume (slice) motion.
The third image in the figure for this section is an example where EDDY was run with inter-slice correction (`--slspec` and `--mporder`) using the slice timing parameter in the sidecar.

In [ ]:
%%bash

mkdir -p "07_eddy_outputs"

# Do EDDY current distortion correction and correct motion between volumes
# Apply TOPUP susceptibility distortion correction
# NOTE:slm=lin if half sphere <-- try if it does not work use tortoise
eddy --imain="06_eddy_inputs/dwi_PAAP.nii.gz" \
    --mask="05_brain_mask/b0_brain_mask.nii.gz" \
    --acqp="06_eddy_inputs/acqparams.txt" --index="06_eddy_inputs/index.txt" \
    --bvecs="06_eddy_inputs/dwi_PAAP.bvec" --bvals="06_eddy_inputs/dwi_PAAP.bval" \
    --topup="04_topup_outputs/topup" --out="07_eddy_outputs/dwi_eddy" \
    --residuals --cnr_maps --data_is_shelled \
    --repol --resamp=jac \
    --niter=8 --fwhm=10,6,4,2,0,0,0,0


In [ ]:
%%bash
fsleyes \
    "06_eddy_inputs/dwi_PAAP.nii.gz" \
    "07_eddy_outputs/dwi_eddy.nii.gz" &

1.
<img src="figures/07a_eddy.gif" width="100%"></img>
*EDDY correction: (a) before eddy showing drop-out; (b) after eddy showing inter-slice motion; (c) example with slice to volume correction.*

## 07b. EDDY QUAD

Generate a quality report for the EDDY output:
- `-idx` and `-par` provide the index and acquisition table. 
- `-m` supplies the mask,
- `-b` the b-values, and `-g` the input b-vectors. 
- `-f` supplies the TOPUP field, 
- `-o` sets the report directory, and `-v` prints details.

Open `07_eddy_quad/eddy.qc/qc.pdf` after running.


In [ ]:
%%bash

# Create mean figures of each shell including b0. 
# Create per-voxel contrast to noise ratio maps for all DWI shells
eddy_quad "07_eddy_outputs/dwi_eddy" \
    -idx "06_eddy_inputs/index.txt" -par "06_eddy_inputs/acqparams.txt" \
    -m "05_brain_mask/b0_brain_mask.nii.gz" \
    -b "06_eddy_inputs/dwi_PAAP.bval" -g "06_eddy_inputs/dwi_PAAP.bvec" \
    -f "04_topup_outputs/topup_field.nii.gz" \
    -o "07_eddy_outputs/dwi_eddy.qc" -v


1. <img src="figures/07b_quad-b0.png" width="100%"></img>
2. <img src="figures/07b_quad-b500.png" width="100%"></img>
3. <img src="figures/07b_quad-b1000.png" width="100%"></img>
4. <img src="figures/07b_quad-b2000.png" width="100%"></img>

*Average corrected diffusion image: (1) b0; (2) b500; (3) b1000; (4) b2000.*

Label all of the images

## 08. Bias field correction

Reduce low frequency intensity variation with `dwibiascorrect ants`:
- `ants` selects ANTs N4 correction
- `-fslgrad` supplies the rotated gradients
- `-mask` restricts estimation to brain
- `-bias` saves the estimated field.

*Note: bias field correction normalizes the b1 field across the full image. For tissue specific models (e.g. MT-CSD) consider tissue specific normalization*

In [ ]:
%%bash

mkdir -p "08_bias_correct"

# Estimate and remove bias field using ANTs N4
dwibiascorrect ants "07_eddy_outputs/dwi_eddy.nii.gz" "08_bias_correct/dwi_n4.nii.gz" \
    -fslgrad "07_eddy_outputs/dwi_eddy.eddy_rotated_bvecs" "06_eddy_inputs/dwi_PAAP.bval" \
    -mask "05_brain_mask/b0_brain_mask.nii.gz" \
    -bias "08_bias_correct/bias.nii.gz"

In [ ]:
%%bash

# Look at the uncorrected, corrected, and residual
fsleyes \
    "07_eddy_outputs/dwi_eddy.nii.gz" \
    "08_bias_correct/dwi_n4.nii.gz"
    "08_bias_correct/bias.nii.gz"

1. <img src="figures/08_bias_correct.gif" width="100%"></img>
2. <img src="figures/08_bias_correct-bias.png" width="100%"></img>

*Bias Correction: (1.a) b0 before correction; (1.b) b0 after correction; (2) estimated bias field.*

## 09. Final mask

Create a mask from the corrected DWI using `-fslgrad` to supply the rotated gradients. Overlay it on the final image.

If your MRtrix version requires an algorithm argument, add `legacy` immediately after `dwi2mask`.


In [ ]:
%%bash

mkdir "09_final_mask"

# create mean b0 to visualize the mask over
dwiextract "08_bias_correct/dwi_n4.nii.gz" - \
    -fslgrad  "07_eddy_outputs/dwi_eddy.eddy_rotated_bvecs" "06_eddy_inputs/dwi_PAAP.bval" \
    -bzero | mrmath - mean "09_final_mask/b0_mean.nii.gz" -axis 3

# create a mask on all volumes and take the consensus
dwi2mask "08_bias_correct/dwi_n4.nii.gz" "09_final_mask/dwi_mask.nii.gz" \
    -fslgrad "07_eddy_outputs/dwi_eddy.eddy_rotated_bvecs" "06_eddy_inputs/dwi_PAAP.bval"

# Create a QC figure of the final mask
"../bin/qc_image_generation.py" \
    -o "09_final_mask/final_mask_qc.png" \
    -i "09_final_mask/b0_mean.nii.gz" \
    -m "09_final_mask/dwi_mask.nii.gz" \
    -x 30 40 50 60 70 80 -y 30 40 50 60 70 80 -z 20 40 60


In [4]:
%%bash

# mkdir -p "09_final_mask"

# # Make a mask by warping the b0 to a template b0 with a mask and reversing the warp.

# # 1. Affine registration, mean b0 -> template
# flirt -ref "b0_template/IITmean_b0.nii.gz" -in "09_final_mask/b0_mean.nii.gz" -omat "09_final_mask/bzero_to_template.mat"

# # 2. Nonlinear registration
# fnirt --ref="b0_template/IITmean_b0.nii.gz" --refmask="b0_template/IITmean_b0_mask.nii.gz" \
#     --in="09_final_mask/b0_mean.nii.gz" --aff="09_final_mask/bzero_to_template.mat" --cout="09_final_mask/bzero_to_template.nii.gz"

# # 3. Invert the warp (subject->template becomes template->subject)
# invwarp --ref="09_final_mask/b0_mean.nii.gz" --warp="09_final_mask/bzero_to_template.nii.gz" --out="09_final_mask/template_to_bzero.nii.gz"

# # 4. Apply the inverted warp to the TEMPLATE's mask
# applywarp --ref="09_final_mask/b0_mean.nii.gz" --in="b0_template/IITmean_b0_mask.nii.gz" \
#     --warp="09_final_mask/template_to_bzero.nii.gz" --out="09_final_mask/b02template.nii.gz"

# 5. Binarize. fslmaths -thr keeps voxels >= threshold (zeroes below it)
fslmaths "09_final_mask/b02template.nii.gz" -bin "09_final_mask/b02template_mask.nii.gz"

# Create a QC figure of the final mask
"../bin/qc_image_generation.py" \
    -o "09_final_mask/b02template_mask_qc.png" \
    -i "09_final_mask/b0_mean.nii.gz" \
    -m "09_final_mask/b02template_mask.nii.gz" \
    -x 30 40 50 60 70 80 -y 30 40 50 60 70 80 -z 20 40 60

In [4]:
%%bash

fsleyes \
    "09_final_mask/b0_mean.nii.gz" \
    "09_final_mask/dwi_mask.nii.gz" \
    "09_final_mask/b02template_mask.nii.gz"

![Add figure](figures/09_final_mask-qc.png)

*Final brain mask from dwi2mask overlaid on the mean b0.*


## 10. Final outputs

Copy the corrected DWI, rotated b-vectors, matching b-values, and final mask into `10_final/`.

Review several slices and diffusion volumes for remaining distortion, dropout, and mask errors.


In [ ]:
%%bash

mkdir -p "10_final"

# Copy the final outputs to one folder
cp -f "08_bias_correct/dwi_n4.nii.gz" "10_final/dwi_final.nii.gz"
cp -f "07_eddy_outputs/dwi_eddy.eddy_rotated_bvecs" "10_final/dwi_final.bvec"
cp -f "06_eddy_inputs/dwi_PAAP.bval" "10_final/dwi_final.bval"
cp -f "09_final_mask/dwi_mask.nii.gz" "10_final/dwi_final_mask.nii.gz"

tree . -I '00_raw_data/DCM_AP|00_raw_data/DCM_PA|11_tract_pipeline'

.
├── 00_raw_data
│   ├── acqparams_AP.txt
│   ├── acqparams_PA.txt
│   ├── dwi_AP.bval
│   ├── dwi_AP.bvec
│   ├── dwi_AP.json
│   ├── dwi_AP.mif
│   ├── dwi_AP.nii.gz
│   ├── dwi_PA.bval
│   ├── dwi_PA.bvec
│   ├── dwi_PA.json
│   ├── dwi_PA.mif
│   ├── dwi_PA.nii.gz
│   ├── dwi_PA_mask.nii.gz
│   ├── index_AP.txt
│   └── index_PA.txt
├── 01_denoise
│   ├── designer-tmp-B349WE
│   │   ├── Npars.nii
│   │   ├── Npars_fixed_strides.nii
│   │   ├── command.txt
│   │   ├── cwd.txt
│   │   ├── dwi.mif
│   │   ├── dwi_designer.bval
│   │   ├── dwi_designer.bvec
│   │   ├── dwi_designer.nii
│   │   ├── dwidn.bval
│   │   ├── dwidn.bvec
│   │   ├── dwidn.mif
│   │   ├── dwidn.nii
│   │   ├── log.txt
│   │   ├── noisemap.mif
│   │   ├── sigma.nii
│   │   ├── sigma_fixed_strides.nii
│   │   ├── working.json
│   │   └── working.mif
│   ├── designer-tmp-H6UBTX
│   │   ├── Npars.nii
│   │   ├── Npars_fixed_strides.nii
│   │   ├── command.txt
│   │   ├── cwd.txt
│   │   ├── dwi.mif
│   │   ├── dwi